# ReviewIQ — Notebook 03: Sentiment Model Training
**Models:** DistilBERT + RoBERTa Ensemble Fusion

This notebook trains the core sentiment classification model:
- Fine-tune `LiYuan/amazon-review-sentiment-analysis` (Amazon-domain warm start) via LoRA
- Fine-tune `roberta-base` on Amazon reviews  
- Build ensemble fusion layer with confidence scoring
- Evaluate per-category performance
- Export to ONNX for fast inference

**Recommended:** GPU node (A10G or better). ~2-3 hours on single GPU.

In [ ]:
!pip install transformers datasets torch accelerate scikit-learn evaluate onnx onnxruntime peft -q

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import classification_report, confusion_matrix
import evaluate

PROCESSED_DIR = Path("../../data/processed")
MODELS_DIR = Path("../../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
CONFIG = {
    "distilbert": {
        "model_name": "distilbert-base-uncased",
        "max_length": 256,
        "batch_size": 64,
        "lr": 3e-5,
        "epochs": 4,
        "warmup_ratio": 0.1,
    },
    "roberta": {
        "model_name": "roberta-base",
        "max_length": 256,
        "batch_size": 32,
        "lr": 2e-5,
        "epochs": 4,
        "warmup_ratio": 0.1,
    },
    "num_labels": 2,  # positive / negative
    "label2id": {"negative": 0, "positive": 1},
    "id2label": {0: "negative", 1: "positive"},
}

# For quick testing, set to True to use 5% of data
QUICK_RUN = False
SAMPLE_FRAC = 0.05 if QUICK_RUN else 1.0
print(f"Quick run: {QUICK_RUN} | Sample fraction: {SAMPLE_FRAC}")

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
train_df = pd.read_parquet(PROCESSED_DIR / "reviews_train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "reviews_val.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "reviews_test.parquet")

if QUICK_RUN:
    train_df = train_df.sample(frac=SAMPLE_FRAC, random_state=42)
    val_df = val_df.sample(frac=SAMPLE_FRAC, random_state=42)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Label distribution (train):")
print(train_df["sentiment_label"].value_counts())

In [ ]:
# ── Dataset class ─────────────────────────────────────────────────────────────
class ReviewDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.texts = df["full_text"].fillna("").tolist()
        self.labels = df["sentiment_label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

In [ ]:
# ── Training function ─────────────────────────────────────────────────────────
def train_model(model_key: str, train_df: pd.DataFrame, val_df: pd.DataFrame):
    cfg = CONFIG[model_key]
    print(f"\n{'='*60}")
    print(f"Training {model_key.upper()} — {cfg['model_name']}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg["model_name"],
        num_labels=CONFIG["num_labels"],
        id2label=CONFIG["id2label"],
        label2id=CONFIG["label2id"],
        ignore_mismatched_sizes=True,  # LiYuan head is 5-class; reinit for binary
    )

    # Apply LoRA for parameter-efficient fine-tuning (DistilBERT leg only)
    if cfg.get("use_lora"):
        from peft import get_peft_model, LoraConfig, TaskType
        lora_cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=cfg["lora_r"],
            lora_alpha=cfg["lora_alpha"],
            lora_dropout=cfg["lora_dropout"],
            target_modules=cfg["lora_target_modules"],
            bias="none",
        )
        model = get_peft_model(model, lora_cfg)
        model.print_trainable_parameters()

    train_ds = ReviewDataset(train_df, tokenizer, cfg["max_length"])
    val_ds = ReviewDataset(val_df, tokenizer, cfg["max_length"])

    output_dir = str(MODELS_DIR / f"{model_key}_sentiment")

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"] * 2,
        learning_rate=cfg["lr"],
        warmup_ratio=cfg["warmup_ratio"],
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none",
        dataloader_num_workers=4,
        gradient_accumulation_steps=2 if model_key == "roberta" else 1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Save final model
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    print(f"\n✅ {model_key} saved to {output_dir}")
    return trainer, tokenizer, model

In [ ]:
# ── Train DistilBERT ──────────────────────────────────────────────────────────
distilbert_trainer, distilbert_tok, distilbert_model = train_model(
    "distilbert", train_df, val_df
)

In [ ]:
# ── Train RoBERTa ─────────────────────────────────────────────────────────────
roberta_trainer, roberta_tok, roberta_model = train_model(
    "roberta", train_df, val_df
)

In [ ]:
# ── Ensemble Fusion ───────────────────────────────────────────────────────────
# Weighted average of softmax probabilities
# RoBERTa gets higher weight (typically more accurate on longer texts)

class EnsembleSentimentModel:
    def __init__(self, distilbert_dir, roberta_dir, weights=(0.4, 0.6)):
        self.weights = weights

        print("Loading DistilBERT...")
        self.db_tok = AutoTokenizer.from_pretrained(distilbert_dir)
        self.db_model = AutoModelForSequenceClassification.from_pretrained(distilbert_dir).to(DEVICE)
        self.db_model.eval()

        print("Loading RoBERTa...")
        self.rb_tok = AutoTokenizer.from_pretrained(roberta_dir)
        self.rb_model = AutoModelForSequenceClassification.from_pretrained(roberta_dir).to(DEVICE)
        self.rb_model.eval()

    def predict(self, texts: list, batch_size=32) -> dict:
        all_probs = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]

            with torch.no_grad():
                # DistilBERT
                db_enc = self.db_tok(batch, truncation=True, padding=True,
                                     max_length=256, return_tensors="pt").to(DEVICE)
                db_logits = self.db_model(**db_enc).logits
                db_probs = torch.softmax(db_logits, dim=-1).cpu().numpy()

                # RoBERTa
                rb_enc = self.rb_tok(batch, truncation=True, padding=True,
                                     max_length=256, return_tensors="pt").to(DEVICE)
                rb_logits = self.rb_model(**rb_enc).logits
                rb_probs = torch.softmax(rb_logits, dim=-1).cpu().numpy()

            # Weighted ensemble
            ensemble_probs = self.weights[0] * db_probs + self.weights[1] * rb_probs
            all_probs.append(ensemble_probs)

        all_probs = np.vstack(all_probs)
        predictions = np.argmax(all_probs, axis=-1)
        confidence = np.max(all_probs, axis=-1)

        return {
            "predictions": predictions,
            "confidence": confidence,
            "positive_prob": all_probs[:, 1],
            "negative_prob": all_probs[:, 0],
            "labels": [CONFIG["id2label"][p] for p in predictions],
        }


ensemble = EnsembleSentimentModel(
    distilbert_dir=str(MODELS_DIR / "distilbert_sentiment"),
    roberta_dir=str(MODELS_DIR / "roberta_sentiment"),
)
print("✅ Ensemble model loaded")

In [ ]:
# ── Evaluate Ensemble on Test Set ─────────────────────────────────────────────
test_texts = test_df["full_text"].fillna("").tolist()
test_labels = test_df["sentiment_label"].astype(int).tolist()

results = ensemble.predict(test_texts)

print("=== Ensemble Test Set Performance ===")
print(classification_report(
    test_labels,
    results["predictions"],
    target_names=["negative", "positive"]
))

# Per-category breakdown
test_df_eval = test_df.copy()
test_df_eval["predicted"] = results["predictions"]
test_df_eval["confidence"] = results["confidence"]
test_df_eval["positive_prob"] = results["positive_prob"]

print("\nPer-category Accuracy:")
cat_acc = test_df_eval.groupby("category").apply(
    lambda g: (g["sentiment_label"] == g["predicted"]).mean()
).round(4)
print(cat_acc.sort_values())

In [ ]:
# ── Save ensemble config ──────────────────────────────────────────────────────
ensemble_config = {
    "distilbert_dir": str(MODELS_DIR / "distilbert_sentiment"),
    "roberta_dir": str(MODELS_DIR / "roberta_sentiment"),
    "weights": [0.4, 0.6],
    "max_length": 256,
    "num_labels": 2,
    "id2label": CONFIG["id2label"],
    "label2id": CONFIG["label2id"],
}

(MODELS_DIR / "ensemble_config.json").write_text(json.dumps(ensemble_config, indent=2))
print("✅ Ensemble config saved")
print(json.dumps(ensemble_config, indent=2))